In [ ]:
%pip -q install -U "openai>=1.70.0" "pydantic>=2.0" pandas tqdm nbformat

## 1. Configuration

In [ ]:
from pathlib import Path
import os, re, json, time, random, hashlib, zipfile, getpass, traceback
from datetime import datetime, timezone
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI
from pydantic import BaseModel, ConfigDict, Field

# Edit these values before running a real experiment.
INPUT_FILE = Path('/...')
OUTPUT_DIR = Path('/content/logsumm_judge_results')
MODEL = 'gpt-6.1-sol'  # Must match checkpoint metadata for resuming.
REASONING_EFFORT = 'medium'
MAX_OUTPUT_TOKENS = 8000
MAX_RETRIES = 3
REQUEST_TIMEOUT_SECONDS = 180
MAX_ROWS = None
DRY_RUN = False

# Set these to None to infer them from the selected prediction file.
DATASET_OVERRIDE = None
SUMMARY_TYPE_OVERRIDE = None
CHUNK_SIZE_OVERRIDE = None

# Resume / recovery options.
UPLOAD_FILES_IN_COLAB = True  # Upload original predictions and checkpoint together.
REQUIRE_CHECKPOINT_FOR_RUN = True  # Safety: refuse a full run without a checkpoint.
RETRY_JUDGE_FAILURES = True
RECOVER_RAW_JUDGMENTS_LOCALLY = True

# Evaluate the ORIGINAL student JSON when prediction-stage validation rejected it.
# Do not fix/reorder/rewrite model output; preserve its original validation error.
EVALUATE_REJECTED_RAW_PREDICTIONS = True

# Preserve truly incomplete/truncated student responses as generation failures.
# NEVER reconstruct JSON or insert/fix events automatically.
AUDIT_REJECTED_STUDENT_GENERATIONS = True

# Recover schema-rejected abstractive JSON only when a real nonempty summary field exists.
# Never promote an explanation into a summary or invent missing generated text.
RECOVER_ABSTRACTIVE_SUMMARY_FIELD = True
REPLACEMENT_PREDICTIONS_FILE = None  # Set to the actual CSV containing corrected student summaries

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Input file:', INPUT_FILE)
print('Dry run:', DRY_RUN, '| max rows:', MAX_ROWS)


## 1a. Upload prediction CSV, checkpoint, and optional repaired predictions



In [ ]:
from pathlib import Path
import shutil

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if UPLOAD_FILES_IN_COLAB:
    try:
        from google.colab import files as colab_files
    except ImportError:
        print('Not in Colab: using the configured file paths.')
    else:
        print('Upload ORIGINAL prediction CSV and matching checkpoint JSONL.')
        print('To evaluate missing_response rows, upload a CSV containing REAL corrected student predictions named *_repaired_predictions.csv.')
        uploaded_files = colab_files.upload()
        candidates = [Path(filename) for filename in uploaded_files]
        checkpoints = [p for p in candidates if p.name.lower().endswith('_judge_checkpoint.jsonl')]
        repair_files = [p for p in candidates if p.name.lower().endswith('_repaired_predictions.csv')]
        prediction_files = [p for p in candidates if p.suffix.lower()=='.csv'
                            and p not in repair_files
                            and not any(term in p.name.lower() for term in
                                        ('_judge_metrics', '_judge_report', '_judge_failures',
                                         '_judge_unresolved_segments', '_judge_input_repair_template'))]
        if len(checkpoints) > 1:
            raise ValueError('Multiple checkpoints uploaded: select only the matching one.')
        if len(repair_files) > 1:
            raise ValueError('Multiple repaired prediction files uploaded: select only one.')
        for cp in checkpoints:
            target = OUTPUT_DIR/cp.name
            if cp.resolve()!=target.resolve(): shutil.copy2(cp,target)
            print('Checkpoint ready:', target)
        if len(prediction_files)==1:
            INPUT_FILE=Path('/content')/prediction_files[0].name
            print('Original input predictions:',INPUT_FILE)
        elif len(prediction_files)>1:
            raise ValueError('Multiple original prediction CSVs uploaded; upload exactly one original predictions file.')
        elif not INPUT_FILE.is_file():
            print('WARNING: Original prediction CSV not present at:', INPUT_FILE)
        if len(repair_files)==1:
            REPLACEMENT_PREDICTIONS_FILE=Path('/content')/repair_files[0].name
            print('Corrected student predictions:',REPLACEMENT_PREDICTIONS_FILE)

print('Selected input:', INPUT_FILE)
print('Replacement predictions:',REPLACEMENT_PREDICTIONS_FILE)
if REPLACEMENT_PREDICTIONS_FILE is None:
    print('NOTE: Without corrected student predictions, missing_response segments cannot be submitted to the judge API.')
print('Checkpoint directory:',OUTPUT_DIR)

## 2. System descriptions and frozen prompt V1.2

In [ ]:
SYSTEM_DESCRIPTIONS = {'bgl': 'IBM BlueGene/L supercomputer logs covering compute nodes, hardware, kernel, and system services. Distinguish actual unsuccessful operations, uncorrected hardware faults, crashes, and disruptions from routine diagnostics, recovered conditions, status reports, and normal maintenance. Interpret what happened from the log evidence, not severity or keyword alone.', 'apache': 'Apache HTTP server. For unknown events, distinguish genuine web-server or application failures from expected request handling and routine operation. Treat an event as a fault only when the logs provide clear evidence of an unsuccessful internal operation, malfunction, or abnormal service behavior. Client-side errors, rejected requests, and diagnostic messages are not faults unless they reveal an underlying system problem. Consider the event context and outcome rather than severity or error keywords alone.', 'linux': 'Linux operating system logs covering kernel, hardware, system services, and user activities. For unknown events, distinguish genuine system malfunctions from routine operations, expected access rejections, and harmless diagnostics. Treat an event as a fault when logs provide clear evidence of an unsuccessful internal operation, system or service malfunction, or abnormal system behavior, including memory exhaustion, forced process termination, and resource allocation failures. Authentication failures, client disconnections, connection timeouts, and access denials are not system faults by themselves unless they reveal a malfunction or disrupted service. Kernel initialization, device configuration, audit records, and normal service lifecycle activities are generally not faults unless they indicate an actual failure. Evaluate the actual outcome and surrounding context rather than error keywords, warnings, or severity alone.', 'openstack': 'OpenStack cloud. Look for failed instance operations and broken service dependencies; normal API requests are background.', 'zookeeper': 'Apache ZooKeeper service. Apache ZooKeeper distributed coordination service managing cluster membership, leader election, client sessions, and replicated data. For unknown events, distinguish genuine coordination or service failures from expected distributed-system behavior, transient connection issues, and routine operations. Treat an event as a fault when logs provide clear evidence of an unsuccessful internal operation, quorum or synchronization failure, service unavailability, data consistency problems, or abnormal server termination. Connection interruptions, election retries, session expirations, socket exceptions, and worker thread shutdowns are not necessarily faults unless they indicate an actual malfunction, failed operation, or disrupted service. Normal leader elections, reconnections, session management, and snapshot operations are generally not faults unless they reveal an underlying failure. Evaluate the actual outcome and surrounding context rather than warnings, exception messages, or severity alone.'}

JUDGE_PROMPT_TEMPLATE = "## ROLE\nYou are an independent evaluator of software log summaries. Evaluate a generated summary using its original log chunk and system description along three dimensions: (1) Faithfulness of atomic factual claims, (2) coverage of distinct source faults, and (3) Conciseness of relevant, nonredundant information.\n\n## SYSTEM CONTEXT\n{SYSTEM_DESCRIPTION}\nUse this context for interpretation, not evidence that a particular event occurred.\n\n## GENERAL RULES\n- Ground all judgments in the supplied log chunk. Do not invent events, causes, impacts, or outcomes.\n- Judge faults from actual observed behavior and outcomes, not severity, warnings, or error keywords alone. Expected rejections, routine diagnostics, successfully corrected conditions, and normal lifecycle activity are not automatically faults.\n- Accept faithful paraphrasing and appropriate abstraction. Infer causal relationships or recovery only when evidenced.\n- Apply identical semantic criteria to extractive, abstractive, and hybrid summaries. Do not penalize a concise accurate normal-operation description or a correctly empty extractive event selection.\n\n## 1. SOURCE ANALYSIS\nIdentify distinct faults (`source_faults`) and informative non-fault events (`source_context`) from the original logs. A fault is an evidenced malfunction, failed operation, resource failure, abnormal termination, or operational disruption. Give exact source event IDs. Group duplicate reports of the same occurrence, but do not merge distinct failures. For context, include meaningful state changes, relevant lifecycle/recovery events, or nearby events that clarify behavior; exclude unimportant repetitive activity. Identify these from the source itself, not from assertions in the summary.\n\n## 2. FAITHFULNESS\nDecompose summary content into atomic factual claims (`atomic_claims`). Classify each distinct claim `supported`, `unsupported`, or `contradicted`, based on source event evidence. Split separable assertions (including causal and outcome claims); do not count the same factual claim twice merely because it appears in different fields. Give brief reasons for non-supported decisions. A chunk-bounded statement of normal operation or no observed faults can be supported when the evidence warrants it.\n\n## 3. FAULT COVERAGE\nFor each identified source fault, determine whether at least one **supported** summary claim communicates the essential fault (`fault_coverage`). Match semantically, not by wording; do not count unsupported speculation or normal context as fault coverage. Link each covered fault to its supporting claim IDs; briefly explain omissions.\n\n## 4. CONCISENESS AND CONTEXT\nIdentify summary information units (`summary_units`), including substantive claims expressed in structured fields or selected log events. Classify each occurrence as:\n- `informative`: supported fault information, meaningful non-fault context, operational outcome, or appropriate concise normal-operation description;\n- `uninformative`: routine, irrelevant, unsupported, or unnecessary information that adds no useful grounded insight;\n- `redundant`: unnecessary repetition of information already conveyed.\nAssess relevance, not length or format. Distinct factual claims are counted once for Faithfulness, while unnecessary repeated occurrences are retained separately for Conciseness. Repeated *source* events may convey a meaningful recurring condition, so describing their recurrence is not inherently redundant. Don't penalize useful identifiers or necessary technical detail. Unsupported assertions are not informative merely because they sound important.\nRecord only materially helpful omitted non-fault context in `missing_important_context`; it is diagnostic and not fault coverage. Set `normal_operation_summary` true only if the summary appropriately communicates normal operation or an intentionally empty event selection with no claimed faults.\n\n## INPUT\nSummary type: {SUMMARY_TYPE}\nOriginal log chunk:\n{LOG_CHUNK}\nGenerated summary:\n{SUMMARY}"
PROMPT_VERSION = "LogSumm-Judge-V1.2"


## 3. Strict structured judge evidence schema

In [ ]:
class StrictModel(BaseModel):
    model_config = ConfigDict(extra='forbid')

class SourceFault(StrictModel):
    fault_id: str
    description: str
    evidence_event_ids: list[str]

class SourceContext(StrictModel):
    context_id: str
    description: str
    evidence_event_ids: list[str]

class AtomicClaim(StrictModel):
    claim_id: str
    claim: str
    classification: str  # enum enforced through schema below
    evidence_event_ids: list[str]
    explanation: str

class FaultCoverage(StrictModel):
    fault_id: str
    covered: bool
    summary_claim_ids: list[str]
    explanation: str

class SummaryUnit(StrictModel):
    unit_id: str
    content: str
    classification: str
    claim_ids: list[str]
    evidence_event_ids: list[str]
    explanation: str

class MissingContext(StrictModel):
    context_id: str
    description: str
    evidence_event_ids: list[str]

class JudgeEvidence(StrictModel):
    source_faults: list[SourceFault]
    source_context: list[SourceContext]
    atomic_claims: list[AtomicClaim]
    fault_coverage: list[FaultCoverage]
    summary_units: list[SummaryUnit]
    missing_important_context: list[MissingContext]
    normal_operation_summary: bool

# Explicit JSON Schema enum constraints while preserving strict Pydantic parsing.
EVIDENCE_SCHEMA = JudgeEvidence.model_json_schema()
EVIDENCE_SCHEMA['$defs']['AtomicClaim']['properties']['classification']['enum'] = ['supported','unsupported','contradicted']
EVIDENCE_SCHEMA['$defs']['SummaryUnit']['properties']['classification']['enum'] = ['informative','uninformative','redundant']
print('Judge output fields:', list(EVIDENCE_SCHEMA['properties']))


## 4. Input adapter: dataset, window size, and summary structure

In [ ]:
KNOWN_DATASETS = tuple(SYSTEM_DESCRIPTIONS)
SUMMARY_TYPES = ('extractive','abstractive','hybrid')
EVENT_ID_PATTERN = re.compile(r'\[id=([^\]]+)\]')
FORBIDDEN_SOURCE_PATTERN = re.compile(r'\[label\s*=|\bevent[_\s-]*template\s*=', re.I)

def infer_settings(path, df):
    filename = path.name.lower()
    dataset = DATASET_OVERRIDE
    if not dataset:
        dataset = next((s for s in sorted(KNOWN_DATASETS,key=len,reverse=True)
                        if re.search(r'(?<![a-z])'+re.escape(s)+r'(?![a-z])',filename)),None)
    if not dataset or dataset not in SYSTEM_DESCRIPTIONS:
        raise ValueError('Cannot infer dataset from filename. Set DATASET_OVERRIDE.')
    stype = SUMMARY_TYPE_OVERRIDE
    if not stype:
        coltypes = [t for t in SUMMARY_TYPES if 'student_'+t in df.columns]
        if len(coltypes)==1: stype=coltypes[0]
        elif 'summary_type' in df and df.summary_type.dropna().nunique()==1:
            v=str(df.summary_type.dropna().iloc[0]).lower()
            if v in SUMMARY_TYPES:stype=v
        if not stype: stype=next((t for t in SUMMARY_TYPES if re.search(r'(?<![a-z])'+t+r'(?![a-z])',filename)),None)
    if stype not in SUMMARY_TYPES: raise ValueError('Unable to infer summary type. Set SUMMARY_TYPE_OVERRIDE.')
    if 'student_'+stype not in df: raise ValueError(f'Missing student_{stype} column.')
    size = CHUNK_SIZE_OVERRIDE
    if size is None:
        m=re.search(r'(?:^|[^a-z0-9])w(\d+)(?=[^a-z0-9]|$)',filename)
        if m:size=int(m.group(1))
        elif 'event_count' in df:
            counts=pd.to_numeric(df.event_count,errors='coerce').dropna().astype(int)
            if len(counts) and counts.nunique()==1:size=int(counts.iloc[0])
        if size is None:
            counts = df.student_input.dropna().astype(str).map(lambda s:len(EVENT_ID_PATTERN.findall(s)))
            if len(counts) and counts.nunique()==1: size=int(counts.iloc[0])
    if size is None: raise ValueError('Cannot infer window size; set CHUNK_SIZE_OVERRIDE.')
    return dataset,stype,int(size)

def parse_student_response(raw, stype):
    """Never read label or teacher columns; parse only the student summary column."""
    if not isinstance(raw,str) or not raw.strip():return None,'missing_response'
    try: obj=json.loads(raw)
    except (TypeError, ValueError):return None,'invalid_json'
    if not isinstance(obj,dict):return None,'invalid_summary_structure'
    explanation=obj.get('explanation','')
    explanation=explanation.strip() if isinstance(explanation,str) else ''
    fields={'extractive':('critical_events','context_events'),
            'abstractive':('summary',),
            'hybrid':('critical_events','context_events','analysis','interpretation')}[stype]
    if any(x not in obj for x in fields):return None,'invalid_summary_structure'
    subset={x:obj[x] for x in fields}
    if stype in ('extractive','hybrid'):
        if any(not isinstance(subset[x],list) or any(not isinstance(t,str) for t in subset[x]) for x in ('critical_events','context_events')):
            return None,'invalid_summary_structure'
    if stype=='abstractive' and not isinstance(subset['summary'],str):return None,'invalid_summary_structure'
    if stype=='hybrid' and not (subset['analysis'] is None or isinstance(subset['analysis'],dict)):
        return None,'invalid_summary_structure'
    if stype=='hybrid' and not (subset['interpretation'] is None or isinstance(subset['interpretation'],str)):
        return None,'invalid_summary_structure'
    # Generation detection: substantive summary OR student explanation means output was produced.
    def has_content(v):
        if v is None:return False
        if isinstance(v,str):return bool(v.strip())
        if isinstance(v,(dict,list)):return any(has_content(q) for q in (v.values() if isinstance(v,dict) else v))
        return True
    if not has_content(subset) and not explanation:return None,'missing_response'
    return subset,'generated'

def validate_source(text):
    if not isinstance(text,str) or not text.strip():raise ValueError('Missing student_input log chunk')
    if FORBIDDEN_SOURCE_PATTERN.search(text):raise ValueError('student_input contains forbidden label or template annotation')
    lines=[l for l in text.splitlines() if l.strip()]
    ids=[EVENT_ID_PATTERN.findall(l) for l in lines]
    if any(len(z)!=1 for z in ids):raise ValueError('Each source line must have exactly one [id=...]')
    flat=[z[0] for z in ids]
    if len(flat)!=len(set(flat)):raise ValueError('Duplicate source event ID')
    return set(flat)

def prepare_frame(path):
    if not path.is_file():raise FileNotFoundError(path)
    df=pd.read_csv(path,dtype=str,low_memory=False,keep_default_na=False)
    if 'student_input' not in df or 'segment_id' not in df:raise ValueError('CSV needs segment_id and student_input')
    dataset,stype,size=infer_settings(path,df)
    if MAX_ROWS is not None:df=df.head(MAX_ROWS).copy()
    if df.segment_id.duplicated().any():raise ValueError('Duplicate segment_id: cannot checkpoint safely')
    print(f'Dataset={dataset}, window={size}, type={stype}, rows={len(df)}')
    return df,dataset,stype,size


## 5. Deterministic JSON validation and metric computation

In [ ]:
def normalize_evidence_ids(judgment):
    """Canonicalize judge-generated local IDs without guessing at missing references.

    Labels are identifiers, not evidence. Preserve the judge's list order and
    update all cross-references; reject duplicate/ambiguous IDs and dangling links.
    """
    d = JudgeEvidence.model_validate(judgment).model_dump()
    specifications = (
        ('source_faults', 'fault_id', 'F'),
        ('source_context', 'context_id', 'K'),
        ('atomic_claims', 'claim_id', 'C'),
        ('summary_units', 'unit_id', 'U'),
    )
    maps = {}
    for collection, field, prefix in specifications:
        mapping = {}
        for index, item in enumerate(d[collection], 1):
            old = item[field].strip()
            if not old or old in mapping:
                raise ValueError(f'Empty or duplicate {field} in {collection}: {old!r}')
            mapping[old] = f'{prefix}{index}'
            item[field] = mapping[old]
        maps[prefix] = mapping

    def remap(value, prefix, where):
        key = value.strip()
        if key not in maps[prefix]:
            raise ValueError(f'Unknown {prefix} reference in {where}: {value!r}')
        return maps[prefix][key]

    for item in d['fault_coverage']:
        item['fault_id'] = remap(item['fault_id'], 'F', 'fault_coverage')
        item['summary_claim_ids'] = [remap(v, 'C', 'fault_coverage') for v in item['summary_claim_ids']]
    for item in d['summary_units']:
        item['claim_ids'] = [remap(v, 'C', 'summary_units') for v in item['claim_ids']]
    # Missing context is diagnostic only; malformed references must not invalidate
    # otherwise scorable faithfulness, fault coverage, or conciseness evidence.
    # Never infer a K reference from a numeric suffix such as M1 -> K1.
    import warnings
    context_by_events = {}
    for ctx in d['source_context']:
        event_key = frozenset(ctx['evidence_event_ids'])
        if event_key:
            context_by_events.setdefault(event_key, []).append(ctx['context_id'])
    cleaned_missing = []
    seen_context_ids = set()
    for item in d['missing_important_context']:
        old_id = item['context_id'].strip()
        if old_id in maps['K']:
            context_id = maps['K'][old_id]
        else:
            candidates = context_by_events.get(frozenset(item['evidence_event_ids']), []) if item['evidence_event_ids'] else []
            if len(candidates) != 1:
                warnings.warn(f'Dropped missing_important_context diagnostic with unresolved reference {old_id!r}; raw judge output is preserved in checkpoint.', stacklevel=2)
                continue
            context_id = candidates[0]
            warnings.warn(f'Normalized missing_important_context reference {old_id!r} to {context_id!r} by unique source-event match.', stacklevel=2)
        if context_id in seen_context_ids:
            warnings.warn(f'Dropped duplicate missing_important_context diagnostic for {context_id!r}.', stacklevel=2)
            continue
        item['context_id'] = context_id
        seen_context_ids.add(context_id)
        cleaned_missing.append(item)
    d['missing_important_context'] = cleaned_missing
    return d

def validate_evidence(judgment, source_ids):
    d=normalize_evidence_ids(judgment)
    def unique_ids(entries,name):
        ids=[e[name] for e in entries]
        if len(ids)!=len(set(ids)):raise ValueError(f'Duplicate {name} values')
        return set(ids)
    fids=unique_ids(d['source_faults'],'fault_id')
    cids=unique_ids(d['atomic_claims'],'claim_id')
    uids=unique_ids(d['summary_units'],'unit_id')
    ctxids=unique_ids(d['source_context'],'context_id')
    for key in ('source_faults','source_context','atomic_claims','summary_units','missing_important_context'):
        for item in d[key]:
            if not set(item['evidence_event_ids']).issubset(source_ids):raise ValueError(f'Unknown event ID in {key}: {item}')
    for item in d['atomic_claims']:
        if item['classification'] not in ('supported','unsupported','contradicted'):raise ValueError('Unknown claim class')
    for item in d['summary_units']:
        if item['classification'] not in ('informative','uninformative','redundant'):raise ValueError('Unknown unit class')
        if not set(item['claim_ids']).issubset(cids):raise ValueError('Unknown summary unit claim ID')
    fmap=d['fault_coverage']
    if len(fmap)!=len(fids) or {v['fault_id'] for v in fmap}!=fids:
        raise ValueError('fault_coverage must contain each source fault exactly once')
    claimmap={c['claim_id']:c for c in d['atomic_claims']}
    for coverage in fmap:
        cl=coverage['summary_claim_ids']
        if not set(cl).issubset(cids):raise ValueError('Fault coverage has unknown claim ID')
        if coverage['covered']:
            if not cl or not any(claimmap[k]['classification']=='supported' for k in cl):
                raise ValueError('Covered fault requires at least one supported summary claim')
        elif cl:raise ValueError('Uncovered fault must have no supporting claim IDs')
    if d['missing_important_context']:
        if len({z['context_id'] for z in d['missing_important_context']})!=len(d['missing_important_context']):
            raise ValueError('Duplicated missing context ID')
        if not {z['context_id'] for z in d['missing_important_context']}.issubset(ctxids):
            raise ValueError('Missing context must refer to a source_context ID')
    return d


# Post-processing version is intentionally separate from the judge prompt version.
# All adjustments below are deterministic and operate only on model-produced evidence.
SCORING_VERSION = 'LogSumm-Scoring-V1.1'

_EMPTY_STRUCTURE_RE = re.compile(
    r'(?i)^\s*(?:'
    r'(?:the\s+)?(?:critical_events|context_events)\s*(?:selection|list|field)?\s*(?:is|are|:)\s*(?:empty|\[\]|none)\.?'
    r'|empty\s+(?:critical_events|context_events)\s*(?:selection|list|field)?\.?'
    r'|(?:the\s+)?(?:critical|context)\s+events?\s+(?:selection|list|field)\s+(?:is|are)\s+empty\.?)\s*$'
)
_NO_FAULT_RE = re.compile(
    r'(?i)\b(?:no|none|without|absence\s+of|not\s+any)\b.{0,55}'
    r'\b(?:faults?|failures?|critical\s+(?:faults?|events?)|errors?|anomal(?:y|ies))\b'
)
_FIELD_INFERENCE_RE = re.compile(
    r'(?i)\b(?:critical_events|context_events|selection\s+as\s+(?:a\s+)?critical\s+event|'
    r'(?:warrant(?:s|ing)?|justif(?:y|ies))\s+(?:its\s+)?selection\s+as\s+(?:a\s+)?critical\s+event|'
    r'classifi(?:ed|cation)\s+as\s+(?:a\s+)?critical\s+event)\b'
)
_EXPLICIT_FAULT_RE = re.compile(
    r'(?i)\b(?:crash(?:ed|es)?|failed|failure|fault|malfunction|outage|'
    r'unavailable|unexpectedly\s+terminated|abnormal(?:ly)?\s+terminat(?:ed|ion)|'
    r'out\s+of\s+memory|memory\s+exhaust(?:ed|ion)|disk\s+full|'
    r'cannot\s+(?:start|write|read|allocate)|unable\s+to\s+(?:start|write|read|allocate))\b'
)

def _summary_has_content(value):
    if value is None: return False
    if isinstance(value, str): return bool(value.strip())
    if isinstance(value, dict): return any(_summary_has_content(v) for v in value.values())
    if isinstance(value, list): return any(_summary_has_content(v) for v in value)
    return True

def _content_strings(value):
    if isinstance(value, str):
        if value.strip(): yield value
    elif isinstance(value, list):
        for child in value: yield from _content_strings(child)
    elif isinstance(value, dict):
        for child in value.values(): yield from _content_strings(child)

def _is_empty_structure_unit(content):
    text = str(content).strip()
    if _EMPTY_STRUCTURE_RE.fullmatch(text): return True
    # Exact representations of the two intentionally empty extractive arrays.
    try:
        obj = json.loads(text)
        return isinstance(obj, dict) and obj and set(obj).issubset({'critical_events', 'context_events'}) and all(v == [] for v in obj.values())
    except (TypeError, ValueError):
        return False

def _is_unasserted_meta_claim(claim, stype, final_summary):
    if stype != 'extractive':
        return False  # Hybrid/abstractive text may explicitly assert interpretations.
    text = claim['claim']
    actual_text = '\n'.join(_content_strings(final_summary))
    # Selection/JSON-key judgments are not factual claims about the source logs.
    if _FIELD_INFERENCE_RE.search(text):
        return True
    # An empty critical_events list is NOT a verbal assertion that there are no faults.
    if _NO_FAULT_RE.search(text) and not _NO_FAULT_RE.search(actual_text):
        return True
    return False

def clean_structural_artifacts(judgment, stype, final_summary):
    """Exclude structural assertions invented from JSON keys/empty lists.

    Preserve substantive text, source-fault analysis, and actual relevance judgments.
    Return audit notes; never manufacture a supported source claim or a missing fault.
    """
    d = normalize_evidence_ids(judgment)
    notes = []
    removed_claim_ids = set()
    for claim in d['atomic_claims']:
        if _is_unasserted_meta_claim(claim, stype, final_summary):
            removed_claim_ids.add(claim['claim_id'])
    if removed_claim_ids:
        d['atomic_claims'] = [c for c in d['atomic_claims'] if c['claim_id'] not in removed_claim_ids]
        notes.append('Removed inferred field/empty-selection claims: ' + ', '.join(sorted(removed_claim_ids)))

    if removed_claim_ids:
        for coverage in d['fault_coverage']:
            coverage['summary_claim_ids'] = [cid for cid in coverage['summary_claim_ids'] if cid not in removed_claim_ids]
            if coverage['covered'] and not coverage['summary_claim_ids']:
                coverage['covered'] = False
                coverage['explanation'] = (coverage['explanation'] or 'No supported summary claim remains after removing structural inference.')
                notes.append('Removed unsupported structural fault coverage for '+coverage['fault_id'])

    # Purely structural content is never an information unit. Where a unit repeats
    # an unasserted no-fault claim, it is also not content actually generated by the student.
    actual_text = '\n'.join(_content_strings(final_summary))
    retained_units=[]
    for u in d['summary_units']:
        u['claim_ids'] = [cid for cid in u['claim_ids'] if cid not in removed_claim_ids]
        content = u['content']
        if _is_empty_structure_unit(content):
            notes.append('Removed empty-JSON structural unit '+u['unit_id'])
            continue
        if (stype == 'extractive' and _NO_FAULT_RE.search(content)
                and not _NO_FAULT_RE.search(actual_text)
                and not u['claim_ids']):
            notes.append('Removed inferred no-fault selection unit '+u['unit_id'])
            continue
        retained_units.append(u)
    d['summary_units'] = retained_units
    # Renumber with link update after removing synthetic claims/units.
    # Distinct IDs remain linked by normalize_evidence_ids.
    return normalize_evidence_ids(d), notes

def _explicit_false_fault_assertion(claim):
    """High-precision detection only: explicit unsupported positive fault assertions.

    Other unsupported details belong to Faithfulness/Conciseness, not Fault
    Completeness. Do not infer a fault simply from an 'error' word or metadata.
    """
    if claim['classification'] == 'supported': return False
    t = claim['claim']
    if _NO_FAULT_RE.search(t): return False
    return bool(_EXPLICIT_FAULT_RE.search(t))

def compute_metrics(d, stype, stripped_summary):
    # Empty summary is judged on its source; generated vs missing response was decided in Python.
    claims=d['atomic_claims']; faults=d['source_faults']; units=d['summary_units']
    good=sum(c['classification']=='supported' for c in claims)
    covered=sum(f['covered'] for f in d['fault_coverage'])
    informative=sum(u['classification']=='informative' for u in units)
    # Vacuous faithfulness: no factual assertions means no hallucinated assertion.
    faithfulness=good/len(claims) if claims else 1.0
    if faults:
        fault_completeness=covered/len(faults)
    else:
        # Fault-free convention. Only explicit unsupported fault assertions
        # trigger the false-fault exception; unrelated unsupported details do not.
        # Ambiguous cases require human review rather than speculative heuristics.
        invented_fault = any(_explicit_false_fault_assertion(c) for c in claims)
        fault_completeness = 0.0 if invented_fault else 1.0
    if units:
        conciseness=informative/len(units)
    else:
        # Intentional zero-content extractive selection is not verbose and
        # should not be penalized for *omitted* non-fault context. Such omissions
        # remain available in missing_important_context diagnostics.
        conciseness=0.0 if faults else 1.0
    return {
        'faithfulness':round(faithfulness,6),
        'fault_completeness':round(fault_completeness,6),
        'conciseness':round(conciseness,6),
        'source_fault_count':len(faults),'covered_fault_count':covered,
        'source_context_count':len(d['source_context']),
        'missing_important_context_count':len(d['missing_important_context']),
        'atomic_claim_count':len(claims),'supported_claim_count':good,
        'unsupported_claim_count':sum(c['classification']=='unsupported' for c in claims),
        'contradicted_claim_count':sum(c['classification']=='contradicted' for c in claims),
        'summary_unit_count':len(units),'informative_unit_count':informative,
        'uninformative_unit_count':sum(u['classification']=='uninformative' for u in units),
        'redundant_unit_count':sum(u['classification']=='redundant' for u in units),
        'fault_containing':bool(faults),
        'normal_operation_summary':d['normal_operation_summary'],
        'possible_invented_fault_claim_count':sum(_explicit_false_fault_assertion(c) for c in claims) if not faults else 0,
    }


## 6. API request, token accounting, and checkpointing

In [ ]:
def build_prompt(dataset, stype, logchunk, summary):
    return JUDGE_PROMPT_TEMPLATE.format(
        SYSTEM_DESCRIPTION=SYSTEM_DESCRIPTIONS[dataset], SUMMARY_TYPE=stype,
        LOG_CHUNK=logchunk,SUMMARY=json.dumps(summary,ensure_ascii=False))

def call_judge(client,prompt):
    response=client.responses.create(
        model=MODEL,
        reasoning={'effort': REASONING_EFFORT},
        input=[{'role':'user','content':prompt}],
        text={'format':{'type':'json_schema','name':'logsumm_judge_evidence',
                        'schema':EVIDENCE_SCHEMA,'strict':True}},
        max_output_tokens=MAX_OUTPUT_TOKENS,
        store=False,
    )
    if response.status!='completed':raise RuntimeError(f'Response status={response.status}, details={response.incomplete_details}')
    if not response.output_text:raise ValueError('Empty model output')
    payload=json.loads(response.output_text)
    usage=response.usage
    details=getattr(usage,'output_tokens_details',None)
    input_details=getattr(usage,'input_tokens_details',None)
    counts={'input_tokens':getattr(usage,'input_tokens',None),
            'output_tokens':getattr(usage,'output_tokens',None),
            'total_tokens':getattr(usage,'total_tokens',None),
            'reasoning_tokens':getattr(details,'reasoning_tokens',None),
            'cached_input_tokens':getattr(input_details,'cached_tokens',None)}
    return payload,counts,response.id

def atomic_jsonl_append(path, obj):
    with path.open('a',encoding='utf-8') as f:
        f.write(json.dumps(obj,ensure_ascii=False,default=str)+'\n')
        f.flush();os.fsync(f.fileno())

def read_checkpoint(path):
    records={}
    if path.exists():
        for lineno,line in enumerate(path.read_text(encoding='utf-8').splitlines(),1):
            if not line.strip():continue
            try:item=json.loads(line)
            except Exception as e:raise RuntimeError(f'Malformed checkpoint line {lineno}: {e}')
            records[str(item['segment_id'])]=item
    return records

def safe_slug(s):return re.sub(r'[^a-z0-9-]+','-',str(s).lower()).strip('-')

def prefix_for(dataset,stype,size):
    return f"{dataset.upper() if dataset=='bgl' else dataset}_w{size}_{stype}"

def build_evidence_boundary_retry_prompt(base_prompt, source_ids, previous_error):
    allowed=sorted((str(x) for x in source_ids), key=lambda x:(not x.isdigit(),int(x) if x.isdigit() else x))
    return (
        base_prompt + '\n\nEVIDENCE-CITATION CORRECTION FOR THIS RETRY ONLY:\n'
        'The previous judgment was rejected because it referenced source event IDs '
        'not present in the supplied log chunk.\n'
        'Use ONLY the exact IDs listed here for every evidence_event_ids field: '
        + json.dumps(allowed,ensure_ascii=False) + '\n'
        'Never extrapolate event IDs from a sequence or cite the next or previous '
        'event outside the chunk. If the source has no evidence supporting a claim, '
        'classify that claim as unsupported rather than inventing an evidence ID.\n'
        'All original metric definitions, schema requirements and source-grounding '
        'rules remain unchanged.\n'
        'Previous validation issue: ' + str(previous_error)[:350]
    )

def unknown_evidence_id_diagnostics(judgment, source_ids):
    # Diagnostics only: this does not modify, repair, or validate the judgment.
    if not isinstance(judgment,dict):return []
    allowed=set(map(str,source_ids))
    problems=[]
    for field in ('source_faults','source_context','atomic_claims','summary_units','missing_important_context'):
        entries=judgment.get(field,[])
        if not isinstance(entries,list):continue
        for idx,entry in enumerate(entries):
            if not isinstance(entry,dict):continue
            ids=entry.get('evidence_event_ids',[])
            if not isinstance(ids,list):continue
            invalid=sorted(set(str(x) for x in ids)-allowed)
            if invalid:problems.append({'field':field,'entry_index':idx,'invalid_event_ids':invalid})
    return problems

def is_unknown_evidence_id_error(error):
    return 'Unknown event ID' in str(error)


## 7. Validate input and preview a request before making API calls

In [ ]:
frame,DATASET,SUMMARY_TYPE,WINDOW_SIZE=prepare_frame(INPUT_FILE)
PREFIX=prefix_for(DATASET,SUMMARY_TYPE,WINDOW_SIZE)
CHECKPOINT_FILE=OUTPUT_DIR/f'{PREFIX}_judge_checkpoint.jsonl'
EVIDENCE_FILE=OUTPUT_DIR/f'{PREFIX}_judge_evidence.jsonl'
METRICS_FILE=OUTPUT_DIR/f'{PREFIX}_judge_metrics.csv'
REPORT_FILE=OUTPUT_DIR/f'{PREFIX}_judge_report.csv'
FAILURES_FILE=OUTPUT_DIR/f'{PREFIX}_judge_failures.csv'
CONFIG_FILE=OUTPUT_DIR/f'{PREFIX}_judge_config.json'
ZIP_FILE=OUTPUT_DIR/f'{PREFIX}_judge_results.zip'

# Optional repaired prediction CSV: only fill missing/invalid original summaries.
# The source logs must match exactly. Never overwrite already valid predictions.
recovered_prediction_ids=[]
if REPLACEMENT_PREDICTIONS_FILE is not None:
    replacement_path=Path(REPLACEMENT_PREDICTIONS_FILE)
    if not replacement_path.is_file(): raise FileNotFoundError(replacement_path)
    replacements=pd.read_csv(replacement_path,dtype=str,low_memory=False,keep_default_na=False)
    summary_col='student_'+SUMMARY_TYPE
    needed={'segment_id','student_input',summary_col}
    if not needed.issubset(replacements.columns):
        raise ValueError(f'Replacement file must have columns {sorted(needed)}')
    if replacements.segment_id.duplicated().any():
        raise ValueError('Duplicate segment_id in replacement CSV')
    replacements=replacements.set_index('segment_id')
    for idx,row in frame.iterrows():
        sid=str(row['segment_id'])
        if sid not in replacements.index:continue
        _, old_status=parse_student_response(row[summary_col],SUMMARY_TYPE)
        if old_status=='generated':continue
        newrow=replacements.loc[sid]
        if str(newrow['student_input'])!=str(row['student_input']):
            raise ValueError(f'Replacement source log differs for segment {sid}; refusing unsafe recovery')
        _,new_status=parse_student_response(newrow[summary_col],SUMMARY_TYPE)
        if new_status=='generated':
            frame.at[idx,summary_col]=newrow[summary_col]
            recovered_prediction_ids.append(sid)
    print('Recovered valid missing/invalid summaries from replacement CSV:',len(recovered_prediction_ids))

raw_recovery_rows=[]
original_rejection_by_id={}

def _original_generation_status(row,stype):
    return str(row.get('student_'+stype+'_status','')).strip().lower()

def recover_abstractive_summary_field(value):
    """Return only a genuinely generated abstractive summary when other keys violate schema.

    Do not rename explanation/exclusion or create a missing summary. An original
    syntactically complete JSON object with a nonempty string summary is required.
    """
    if not RECOVER_ABSTRACTIVE_SUMMARY_FIELD or not isinstance(value,str) or not value.strip():
        return None
    try:
        obj=json.loads(value)
    except (ValueError,TypeError):
        return None
    if not isinstance(obj,dict):
        return None
    summary=obj.get('summary')
    return {'summary':summary} if isinstance(summary,str) and summary.strip() else None

def recover_original_student_json(row, stype):
    if not EVALUATE_REJECTED_RAW_PREDICTIONS:
        return None,None,'recovery_disabled'
    if _original_generation_status(row,stype) not in ('failed','invalid','validation_failed'):
        return None,None,'not_rejected'
    observations=[]
    for col in ('student_'+stype+'_json_extracted','student_'+stype+'_raw'):
        value=row.get(col,'')
        parsed,status=parse_student_response(value,stype)
        if status=='generated':
            return parsed,col,'complete_generated_json'
        if stype=='abstractive':
            recovered=recover_abstractive_summary_field(value)
            if recovered is not None:
                return recovered,col,'original_summary_field_from_schema_rejected_json'
        observations.append(status)
    if any(s=='invalid_json' for s in observations):
        return None,None,'truncated_or_unparseable_json'
    if any(s=='invalid_summary_structure' for s in observations):
        return None,None,'incomplete_summary_structure'
    return None,None,'no_usable_generated_json'

prepared=[]
for _,row in frame.iterrows():
    sid=str(row['segment_id']); source=row['student_input']
    generation_status=_original_generation_status(row,SUMMARY_TYPE)
    student_error=str(row.get('student_'+SUMMARY_TYPE+'_error',''))
    rejected=generation_status in ('failed','invalid','validation_failed')
    raw_source=''; outcome='not_rejected'; input_status='ready'
    try:
        source_ids=validate_source(source)
        summary,status=parse_student_response(row['student_'+SUMMARY_TYPE],SUMMARY_TYPE)
        input_status=status
        if summary is None:
            raw_summary,raw_source,outcome=recover_original_student_json(row,SUMMARY_TYPE)
            if raw_summary is not None:
                summary=raw_summary
                status='generated'
        if summary is None:
            prepared.append({'segment_id':sid,'status':status,'summary':None,'logchunk':source,'source_ids':source_ids})
        else:
            prepared.append({'segment_id':sid,'status':'ready','summary':summary,'logchunk':source,'source_ids':source_ids})
    except Exception as exc:
        status='input_error';outcome='source_input_error'
        prepared.append({'segment_id':sid,'status':status,'error':str(exc),'summary':None,'logchunk':'','source_ids':set()})
    if rejected and AUDIT_REJECTED_STUDENT_GENERATIONS:
        raw_recovery_rows.append({
            'segment_id':sid,
            'original_student_status':generation_status,
            'original_student_error':student_error,
            'original_input_status':input_status,
            'recovered_from_column':raw_source or '',
            'recovery_classification':outcome,
            'judge_input_status':prepared[-1]['status'],
            'action':('judge_original_unmodified_student_json' if raw_source else
                      'retain_as_original_generation_failure' if prepared[-1]['status']!='ready' else
                      'use_original_validated_student_summary'),
        })
        original_rejection_by_id[sid]=raw_recovery_rows[-1]

RAW_RECOVERY_AUDIT_FILE=OUTPUT_DIR/f'{PREFIX}_judge_raw_prediction_recovery_audit.csv'
pd.DataFrame(raw_recovery_rows,columns=[
    'segment_id','original_student_status','original_student_error',
    'original_input_status','recovered_from_column','recovery_classification','judge_input_status','action'
]).to_csv(RAW_RECOVERY_AUDIT_FILE,index=False)
recovered_raw_rows=[r for r in raw_recovery_rows if r['recovered_from_column']]
unrecoverable_raw_rows=[r for r in raw_recovery_rows if r['judge_input_status']!='ready']
print('Original prediction-stage rejections:',len(raw_recovery_rows))
print('Complete generated JSON or original summary field recovered for judgment:',len(recovered_raw_rows))
print('Rejected predictions still not judgeable:',len(unrecoverable_raw_rows))
if raw_recovery_rows:
    print('Recovery classes:',pd.Series([r['recovery_classification'] for r in raw_recovery_rows]).value_counts().to_dict())
    print('Diagnostic CSV:',RAW_RECOVERY_AUDIT_FILE)
if unrecoverable_raw_rows:
    print('Unjudgeable IDs:',[r['segment_id'] for r in unrecoverable_raw_rows])
    print('These require original student inference recovery; JSON and missing summary fields are never synthetically completed.')
print('Input statuses:',pd.Series([r['status'] for r in prepared]).value_counts().to_dict())
example=next((v for v in prepared if v['status']=='ready'),None)
if example:
    preview_prompt=build_prompt(DATASET,SUMMARY_TYPE,example['logchunk'],example['summary'])
    print('Example segment:',example['segment_id'])
    print('Original log preview:',example['logchunk'][:350])
    print('Summary without explanation:',json.dumps(example['summary'],ensure_ascii=False)[:700])
    assert '"explanation"' not in json.dumps(example['summary'])
else:print('No valid generated summaries to evaluate')


### 7a. Identify missing student predictions and create a repair template




In [ ]:
# This diagnostic does not use the OpenAI API.
missing_items=[item for item in prepared if item['status'] != 'ready']
print('Segments without judgeable student inputs:',len(missing_items))
if missing_items:
    status_summary=pd.Series([item['status'] for item in missing_items]).value_counts()
    print('Reasons:',status_summary.to_dict())
    print('Segment IDs / reasons:')
    display(pd.DataFrame([{'segment_id':item['segment_id'],
                           'input_status':item['status'],
                           'detail':item.get('error','')}
                          for item in missing_items]))
    repair_col='student_'+SUMMARY_TYPE
    affected_ids={item['segment_id'] for item in missing_items}
    template_rows=frame[frame.segment_id.astype(str).isin(affected_ids)][
        ['segment_id','student_input',repair_col]].copy()
    template_rows.insert(1,'input_status',template_rows['segment_id'].map(
        {item['segment_id']:item['status'] for item in missing_items}))
    REPAIR_TEMPLATE_FILE=OUTPUT_DIR/f'{PREFIX}_judge_input_repair_template.csv'
    template_rows.to_csv(REPAIR_TEMPLATE_FILE,index=False)
    print('Repair template:',REPAIR_TEMPLATE_FILE)
    print('IMPORTANT: Re-running the judge alone cannot repair these inputs.')
    print('Supply real corrected student model predictions in the template before re-running.')
    print('Save a populated copy named',PREFIX+'_repaired_predictions.csv', 'and upload it with the original CSV and checkpoint.')
else:
    REPAIR_TEMPLATE_FILE=None
    print('All student input summaries are valid and judgeable.')

## 8. Execute (or dry run) and save resumable judgments

In [ ]:
prompt_hash=hashlib.sha256((JUDGE_PROMPT_TEMPLATE+json.dumps(SYSTEM_DESCRIPTIONS,sort_keys=True)+json.dumps(EVIDENCE_SCHEMA,sort_keys=True)).encode()).hexdigest()
file_hash=hashlib.sha256(INPUT_FILE.read_bytes()).hexdigest()
config={'dataset':DATASET,'window_size':WINDOW_SIZE,'summary_type':SUMMARY_TYPE,'model':MODEL,'reasoning_effort':REASONING_EFFORT,
        'prompt_version':PROMPT_VERSION,'prompt_sha256':prompt_hash,'input_file':str(INPUT_FILE),
        'input_file_sha256':file_hash,'row_limit':MAX_ROWS,'dry_run':DRY_RUN,
        'max_output_tokens':MAX_OUTPUT_TOKENS,'scoring_version':SCORING_VERSION,'recover_abstractive_summary_field':RECOVER_ABSTRACTIVE_SUMMARY_FIELD,'created_utc':datetime.now(timezone.utc).isoformat(),
        'system_description':SYSTEM_DESCRIPTIONS[DATASET]}
# Verify the checkpoint before any paid API calls. Keep the original CSV hash for
# compatibility, even when optional replacement summaries are used in memory.
existing=read_checkpoint(CHECKPOINT_FILE)
if REQUIRE_CHECKPOINT_FOR_RUN and not existing and not DRY_RUN:
    raise RuntimeError(f'No usable checkpoint at {CHECKPOINT_FILE}. Refusing to restart all segments. Upload the matching checkpoint.')
if existing:
    for record in existing.values():
        if (record.get('model')!=MODEL or
            record.get('reasoning_effort')!=REASONING_EFFORT or
            record.get('prompt_sha256')!=prompt_hash or
            record.get('input_file_sha256')!=file_hash):
            raise RuntimeError('Checkpoint belongs to a different model, prompt, or original input CSV. Verify your configuration and selected files.')

# Match IDs as strings, and only trust completed records with a saved judgment/metrics.
input_ids={item['segment_id'] for item in prepared}
unknown=set(existing)-input_ids
if unknown:
    raise RuntimeError(f'Checkpoint contains {len(unknown)} IDs absent from the input, e.g. {list(sorted(unknown))[:5]}')

def needs_evaluation(item):
    rec=existing.get(item['segment_id'])
    if item['status']!='ready':
        if rec is not None and rec.get('status')=='completed':
            raise RuntimeError(f"Input for previously completed segment {item['segment_id']} is now invalid. Verify the original predictions file.")
        return rec is None or rec.get('status')!=item['status']
    if rec is None:return True
    if rec.get('status')=='completed' and rec.get('judgment') and rec.get('metrics'):
        return False
    if rec.get('status')=='judge_failure':return RETRY_JUDGE_FAILURES
    # An earlier missing/invalid input is now usable after CSV repair.
    if rec.get('status') in ('missing_response','invalid_json','invalid_summary_structure','input_error'):
        return True
    return True

# Judge calls require valid student summaries. The checkpoint's missing_response
# records are not successful evaluations and must remain visible in the audit.
pending=[item for item in prepared if item['status']=='ready' and needs_evaluation(item)]
unjudgeable=[item for item in prepared if item['status']!='ready']
completed_ids={sid for sid,r in existing.items() if r.get('status')=='completed' and r.get('judgment') and r.get('metrics')}
print('===== PRE-RUN CHECKPOINT AUDIT =====')
print('Total input segments:',len(prepared))
print('Unique checkpoint records:',len(existing))
print('Checkpoint statuses:',pd.Series([r.get('status') for r in existing.values()]).value_counts().to_dict() if existing else {})
print('Previously completed and reusable judgments:',len(completed_ids))
print('Recovered student predictions from replacement CSV:',len(recovered_prediction_ids))
print('Original rejected raw JSON predictions now judgeable:',len(recovered_raw_rows))
print('Original rejected predictions still unjudgeable:',len(unrecoverable_raw_rows))
print('Pending judge evaluations (ready for API/local recovery):',len(pending))
print('UNRESOLVED missing/invalid student predictions:',len(unjudgeable))
if unjudgeable:
    print('Input failure reasons:',pd.Series([r['status'] for r in unjudgeable]).value_counts().to_dict())
    print('Input failure segment IDs:',[r['segment_id'] for r in unjudgeable])
    print('ACTION REQUIRED: These records cannot be judged without actual student predictions.')
    print('Fill student_'+SUMMARY_TYPE+' in the generated *_judge_input_repair_template.csv,')
    print('save as *_repaired_predictions.csv, upload it, then rerun preparation and evaluation.')
    if REPLACEMENT_PREDICTIONS_FILE is None:
        print('No replacement predictions file supplied; NO judge API request will be made for these records.')
if not pending:
    if unjudgeable:
        print('NO JUDGE REQUESTS POSSIBLE: all remaining unresolved rows lack judgeable student input.')
    else:
        print('ALL SEGMENTS ALREADY COMPLETED; no API requests needed.')
print('Previously successful evaluations are skipped.')
if pending:
    print('Pending segment IDs:',[item['segment_id'] for item in pending])

if DRY_RUN:
    print('DRY_RUN=True: no API calls.')
else:
    if any(item['status']=='ready' and not (RECOVER_RAW_JUDGMENTS_LOCALLY and existing.get(item['segment_id'],{}).get('raw_judgment')) for item in pending):
        if not os.environ.get('OPENAI_API_KEY'):
            try:
                from google.colab import userdata
                secret=userdata.get('OPENAI_API_KEY')
                if secret:os.environ['OPENAI_API_KEY']=secret
            except (ImportError,KeyError,Exception):
                pass
        if not os.environ.get('OPENAI_API_KEY'):
            os.environ['OPENAI_API_KEY']=getpass.getpass('OpenAI API key (hidden): ')
        client=OpenAI(api_key=os.environ['OPENAI_API_KEY'],timeout=REQUEST_TIMEOUT_SECONDS,max_retries=0)
    else:
        client=None
    for item in tqdm(pending,desc='Unresolved judge segments'):
        sid=item['segment_id']
        record={'segment_id':sid,'dataset':DATASET,'window_size':WINDOW_SIZE,
                'summary_type':SUMMARY_TYPE,'model':MODEL,'reasoning_effort':REASONING_EFFORT,'prompt_version':PROMPT_VERSION,
                'prompt_sha256':prompt_hash,'input_file_sha256':file_hash,
                'evaluated_utc':datetime.now(timezone.utc).isoformat(),
                'status':item['status'],'error':'', 'input_tokens':None,
                'output_tokens':None,'total_tokens':None,'reasoning_tokens':None,
                'cached_input_tokens':None,'inference_time_seconds':0.0,
                'api_attempts':0,'response_id':None,'judgment':None,'raw_judgment':None,'metrics':None,
                'scoring_version':SCORING_VERSION,'postprocessing_notes':[],
                'failed_judge_attempts':[]}
        if item['status']=='ready':
            old=existing.get(sid)
            if RECOVER_RAW_JUDGMENTS_LOCALLY and old and old.get('raw_judgment'):
                try:
                    raw=old['raw_judgment']
                    checked=validate_evidence(raw,item['source_ids'])
                    checked,notes=clean_structural_artifacts(checked,SUMMARY_TYPE,item['summary'])
                    checked=validate_evidence(checked,item['source_ids'])
                    record.update({k:old.get(k) for k in ('input_tokens','output_tokens','total_tokens',
                                   'reasoning_tokens','cached_input_tokens','inference_time_seconds',
                                   'api_attempts','response_id')})
                    record.update(raw_judgment=raw,judgment=checked,postprocessing_notes=notes,
                                  metrics=compute_metrics(checked,SUMMARY_TYPE,item['summary']),status='completed')
                    atomic_jsonl_append(CHECKPOINT_FILE,record)
                    existing[sid]=record
                    continue
                except Exception as exc:
                    print(f'Segment {sid}: saved raw judgment could not be locally recovered ({type(exc).__name__}: {exc}); requesting new judgment.')
                    if client is None:
                        if not os.environ.get('OPENAI_API_KEY'):
                            os.environ['OPENAI_API_KEY']=getpass.getpass('OpenAI API key (hidden): ')
                        client=OpenAI(api_key=os.environ['OPENAI_API_KEY'],timeout=REQUEST_TIMEOUT_SECONDS,max_retries=0)
            prompt=build_prompt(DATASET,SUMMARY_TYPE,item['logchunk'],item['summary'])
            last_err=None
            # Existing checkpoint judge failures can immediately receive a targeted
            # event-boundary retry. Do not rejudge completed records.
            boundary_retry_needed=bool(old and old.get('status')=='judge_failure' and
                                       (is_unknown_evidence_id_error(old.get('error','')) or
                                        any(is_unknown_evidence_id_error(a.get('error',''))
                                            for a in old.get('failed_judge_attempts',[]))))
            previous_boundary_error=old.get('error','') if boundary_retry_needed else ''
            if boundary_retry_needed:
                for earlier in old.get('failed_judge_attempts',[]):
                    if is_unknown_evidence_id_error(earlier.get('error','')):
                        record['failed_judge_attempts'].append({
                            'attempt':'previous_checkpoint',
                            'response_id':earlier.get('response_id'),
                            'error':earlier.get('error',''),
                            'invalid_evidence_references':earlier.get('invalid_evidence_references')
                            or unknown_evidence_id_diagnostics(earlier.get('raw_judgment'),item['source_ids']),
                            'boundary_retry_guidance_used':False})
                if old.get('raw_judgment') and is_unknown_evidence_id_error(old.get('error','')):
                    record['failed_judge_attempts'].append({
                        'attempt':'previous_checkpoint','response_id':old.get('response_id'),
                        'error':old.get('error',''),
                        'invalid_evidence_references':unknown_evidence_id_diagnostics(
                            old.get('raw_judgment'),item['source_ids']),
                        'boundary_retry_guidance_used':False})
            for attempt in range(1,MAX_RETRIES+1):
                t0=time.perf_counter()
                try:
                    active_prompt=(build_evidence_boundary_retry_prompt(prompt,item['source_ids'],previous_boundary_error)
                                   if boundary_retry_needed else prompt)
                    evidence,usage,response_id=call_judge(client,active_prompt)
                    elapsed=time.perf_counter()-t0
                    record['inference_time_seconds']+=elapsed
                    record['api_attempts']=attempt
                    record.update(usage)
                    record['response_id']=response_id
                    record['raw_judgment']=evidence
                    # Reject hallucinated evidence IDs; the judge must regenerate.
                    # Validation rules are intentionally NOT relaxed.
                    checked=validate_evidence(evidence,item['source_ids'])
                    checked,notes=clean_structural_artifacts(checked,SUMMARY_TYPE,item['summary'])
                    checked=validate_evidence(checked,item['source_ids'])
                    record['postprocessing_notes']=notes
                    record['judgment']=checked
                    record['metrics']=compute_metrics(checked,SUMMARY_TYPE,item['summary'])
                    record['status']='completed';last_err=None;break
                except Exception as exc:
                    elapsed=time.perf_counter()-t0
                    record['inference_time_seconds']+=elapsed
                    record['api_attempts']=attempt
                    last_err=f'{type(exc).__name__}: {exc}'
                    # A returned response can still fail evidence validation (e.g.,
                    # citation of a nonexistent source event). Retry with a NEW API
                    # request rather than treating this as unrecoverable. Preserve
                    # every rejected judge response in diagnostics.
                    if record['raw_judgment'] is not None:
                        record['failed_judge_attempts'].append({
                            'attempt':attempt,'response_id':record['response_id'],
                            'error':last_err,'raw_judgment':record['raw_judgment'],
                            'invalid_evidence_references':unknown_evidence_id_diagnostics(
                                record['raw_judgment'],item['source_ids']),
                            'boundary_retry_guidance_used':boundary_retry_needed})
                        if is_unknown_evidence_id_error(last_err):
                            boundary_retry_needed=True
                            previous_boundary_error=last_err
                        record['raw_judgment']=None
                        record['judgment']=None
                    # Quota failures are not recoverable by immediate retry.
                    if ('insufficient_quota' in last_err or 'no credits remaining' in last_err.lower()
                        or 'exceeded your current quota' in last_err.lower()):break
                    if attempt<MAX_RETRIES:time.sleep(min(2**attempt+random.random(),12))
            if last_err:
                record['status']='judge_failure';record['error']=last_err
        elif item['status']=='input_error':record['error']=item.get('error','')
        atomic_jsonl_append(CHECKPOINT_FILE,record)
        existing[sid]=record
        if ('insufficient_quota' in record.get('error','')
            or 'no credits remaining' in record.get('error','').lower()
            or 'exceeded your current quota' in record.get('error','').lower()):
            print('API credits exhausted; stopping without submitting the remaining requests.')
            break
    print('Judge execution finished. Checkpoint:',CHECKPOINT_FILE)
    if any(existing.get(i['segment_id'],{}).get('failed_judge_attempts') for i in pending):
        print('Judge responses rejected by evidence validation are stored in failed_judge_attempts for review.')
    if unjudgeable:
        print(f'WARNING: {len(unjudgeable)} records still have missing/invalid STUDENT predictions and were NOT judged.')


## 9. Export detailed evidence, per-chunk metrics, aggregate report, and failures

In [ ]:
records=read_checkpoint(CHECKPOINT_FILE)
# Recompute existing successful results whenever local deterministic scoring changes.
# Checkpoint is append-only; raw evidence and original API usage stay untouched.
for prepared_item in prepared:
    sid=prepared_item['segment_id']
    rec=records.get(sid)
    if not rec or rec.get('status')!='completed' or rec.get('scoring_version')==SCORING_VERSION:
        continue
    original=rec.get('raw_judgment') or rec.get('judgment')
    if not original:
        raise ValueError(f'Cannot rescore completed checkpoint {sid}: missing judgment')
    checked=validate_evidence(original,prepared_item['source_ids'])
    checked,notes=clean_structural_artifacts(checked,SUMMARY_TYPE,prepared_item['summary'])
    checked=validate_evidence(checked,prepared_item['source_ids'])
    updated={**rec,'judgment':checked,
             'metrics':compute_metrics(checked,SUMMARY_TYPE,prepared_item['summary']),
             'scoring_version':SCORING_VERSION,'postprocessing_notes':notes}
    atomic_jsonl_append(CHECKPOINT_FILE,updated)
    records[sid]=updated
ordered=[]
for item in prepared:
    sid=item['segment_id']
    if sid in records:ordered.append(records[sid])
    else:
        ordered.append({'segment_id':sid,'dataset':DATASET,'window_size':WINDOW_SIZE,
                        'summary_type':SUMMARY_TYPE,'status':'dry_run' if DRY_RUN and item['status']=='ready' else 'not_evaluated' if item['status']=='ready' else item['status'],
                        'error':item.get('error',''),'judgment':None,'metrics':None})
with EVIDENCE_FILE.open('w',encoding='utf-8') as f:
    for rec in ordered:
        f.write(json.dumps(rec,ensure_ascii=False,default=str)+'\n')
metric_rows=[];errors=[]
for rec in ordered:
    row={k:rec.get(k) for k in ('segment_id','dataset','window_size','summary_type','status','model','input_tokens','output_tokens','total_tokens','reasoning_tokens','cached_input_tokens','inference_time_seconds','api_attempts','response_id')}
    row.update(rec.get('metrics') or {})
    metric_rows.append(row)
    if rec['status'] not in ('completed','dry_run'):
        errors.append({**row,'error':rec.get('error','')})
metrics_df=pd.DataFrame(metric_rows)
metrics_df.to_csv(METRICS_FILE,index=False)
pd.DataFrame(errors,columns=list(metric_rows[0])+['error'] if metric_rows else ['segment_id','status','error']).to_csv(FAILURES_FILE,index=False)
valid=metrics_df[metrics_df.status=='completed'].copy()
numcols=['faithfulness','fault_completeness','conciseness']
report={'dataset':DATASET,'window_size':WINDOW_SIZE,'summary_type':SUMMARY_TYPE,'model':MODEL,
        'input_file':INPUT_FILE.name,'total_rows':len(metrics_df),
        'completed':len(valid),'generation_or_input_failures':int(metrics_df.status.isin(['missing_response','invalid_json','invalid_summary_structure','input_error']).sum()),
        'judge_failures':int((metrics_df.status=='judge_failure').sum()),
        'original_student_validation_failures':len(raw_recovery_rows),
        'original_rejected_json_recovered_for_judging':len(recovered_raw_rows),
        'original_rejected_generations_still_unjudgeable':len(unrecoverable_raw_rows),
        'input_tokens_total':pd.to_numeric(valid.get('input_tokens',pd.Series(dtype=float)),errors='coerce').sum(),
        'output_tokens_total':pd.to_numeric(valid.get('output_tokens',pd.Series(dtype=float)),errors='coerce').sum(),
        'inference_time_seconds_total':pd.to_numeric(valid.get('inference_time_seconds',pd.Series(dtype=float)),errors='coerce').sum()}
for metric in numcols:
    report[metric+'_mean_all_valid']=pd.to_numeric(valid.get(metric,pd.Series(dtype=float)),errors='coerce').mean()
    report[metric+'_std_all_valid']=pd.to_numeric(valid.get(metric,pd.Series(dtype=float)),errors='coerce').std()
if not valid.empty and 'fault_containing' in valid:
    subset=valid[valid.fault_containing==True]
    report['fault_containing_chunks']=len(subset)
    report['fault_completeness_mean_fault_containing']=pd.to_numeric(subset.fault_completeness,errors='coerce').mean()
    report['fault_containing_total_faults']=pd.to_numeric(subset.source_fault_count,errors='coerce').sum()
    report['fault_containing_total_covered_faults']=pd.to_numeric(subset.covered_fault_count,errors='coerce').sum()
else:
    report['fault_containing_chunks']=0
report['input_tokens_mean_per_valid']=pd.to_numeric(valid.get('input_tokens',pd.Series(dtype=float)),errors='coerce').mean()
report['output_tokens_mean_per_valid']=pd.to_numeric(valid.get('output_tokens',pd.Series(dtype=float)),errors='coerce').mean()
report['inference_time_seconds_mean_per_valid']=pd.to_numeric(valid.get('inference_time_seconds',pd.Series(dtype=float)),errors='coerce').mean()
report['prompt_version']=PROMPT_VERSION
report['prompt_sha256']=prompt_hash
pd.DataFrame([report]).to_csv(REPORT_FILE,index=False)
CONFIG_FILE.write_text(json.dumps(config,indent=2,ensure_ascii=False),encoding='utf-8')
print(pd.DataFrame([report]).T.to_string(header=False))
print('Saved:',[x.name for x in (EVIDENCE_FILE,METRICS_FILE,REPORT_FILE,FAILURES_FILE,CONFIG_FILE)])


## 10. Download all outputs as a ZIP file

In [ ]:
# Audit every input ID, including rows skipped due to quota exhaustion.
records=read_checkpoint(CHECKPOINT_FILE)
prepared_by_id={item['segment_id']:item for item in prepared}
audit_rows=[]
for item in prepared:
    sid=item['segment_id'];rec=records.get(sid)
    status=rec.get('status') if rec else 'not_evaluated'
    # An old input-failure record is not completion for a now-repaired summary.
    if item['status']=='ready' and status in ('missing_response','invalid_json',
                                             'invalid_summary_structure','input_error'):
        status='not_evaluated'
    if item['status']!='ready' and status=='completed':
        status='input_changed_review_required'
    if status=='completed' and (not rec.get('judgment') or not rec.get('metrics')):
        status='incomplete_checkpoint_record'
    if status!='completed':
        audit_rows.append({'segment_id':sid,'status':status,
                           'input_status':item['status'],
                           'error':(rec or {}).get('error',item.get('error','')),
                           'has_raw_judgment':bool((rec or {}).get('raw_judgment')),
                           'original_student_validation_status':original_rejection_by_id.get(sid,{}).get('original_student_status',''),
                           'original_student_validation_error':original_rejection_by_id.get(sid,{}).get('original_student_error',''),
                           'recovery_classification':original_rejection_by_id.get(sid,{}).get('recovery_classification',''),
                           'possible_next_action':(
                               'Supply repaired student prediction' if item['status']!='ready' else
                               'Recover/retry judge response' if status=='judge_failure' else
                               'Run pending evaluation')})
AUDIT_FILE=OUTPUT_DIR/f'{PREFIX}_judge_unresolved_segments.csv'
audit_columns=['segment_id','status','input_status','error','has_raw_judgment','original_student_validation_status','original_student_validation_error','recovery_classification','possible_next_action']
pd.DataFrame(audit_rows,columns=audit_columns).to_csv(AUDIT_FILE,index=False)
print('===== FINAL COMPLETENESS AUDIT =====')
print('Total input segments:',len(prepared))
print('Completed judgments:',len(prepared)-len(audit_rows))
print('Unresolved segments:',len(audit_rows))
print('Completion rate: {:.1%}'.format((len(prepared)-len(audit_rows))/len(prepared)) if prepared else 'No input segments')
print('Breakdown:',dict(pd.Series([r['status'] for r in audit_rows]).value_counts()) if audit_rows else {})
if audit_rows:
    print('INCOMPLETE: some segments remain unresolved. Full list:',AUDIT_FILE)
    print('A judge cannot reconstruct truncated/invalid original student generations.')
    display(pd.DataFrame(audit_rows).head(20))
    input_unresolved=sum(r['input_status']!='ready' for r in audit_rows)
    if input_unresolved:
        print(f'{input_unresolved} unresolved rows have missing/invalid STUDENT summaries (not judge API failures).')
        print('To complete them: regenerate/recover their student predictions, then upload the corrected *_repaired_predictions.csv and rerun.')
else:
    print('COMPLETE: every input segment has a validated, scored judge result.')
print('Original prediction-stage validation failures (even if subsequently judged):',len(raw_recovery_rows))
print('Originally rejected outputs recovered for judging:',len(recovered_raw_rows))
print('Original student generations still unjudgeable:',len(unrecoverable_raw_rows))
if len(audit_rows)==0 and raw_recovery_rows:
    print('NOTE: Judge-complete does not mean every original student prediction passed extractive/hybrid validation.')


In [ ]:
outputs=[EVIDENCE_FILE,METRICS_FILE,REPORT_FILE,FAILURES_FILE,CONFIG_FILE,AUDIT_FILE,RAW_RECOVERY_AUDIT_FILE]
if CHECKPOINT_FILE.exists():outputs.append(CHECKPOINT_FILE)
if REPAIR_TEMPLATE_FILE is not None and REPAIR_TEMPLATE_FILE.exists():outputs.append(REPAIR_TEMPLATE_FILE)
with zipfile.ZipFile(ZIP_FILE,'w',compression=zipfile.ZIP_DEFLATED) as zf:
    for p in outputs:
        if p.is_file():zf.write(p,arcname=p.name)
print('ZIP archive:',ZIP_FILE,'| files:',len(outputs))
try:
    from google.colab import files
    files.download(str(ZIP_FILE))
except ImportError:
    print('Non-Colab environment: ZIP is available at the printed path.')
